# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #4, "The Freshness Multiplier"** (`docs/flyrank-seo-research-march-2026.pdf`, p.9). The
headline claim: 365+ day content refreshed within 30 days shows a 3.2x health boost (10.7 -> 34.5) and
57x more impressions (71 -> 4039).

**My methodology question:** where does the "refreshed" group come from -- was it chosen at random, or
is it whichever old pages someone already decided were worth refreshing? If it's the latter, the pages
picked for a refresh were probably already showing some residual signal (real search demand, an editor's
judgment that it was worth the effort) that a truly abandoned page wouldn't have. That would inflate the
apparent effect of refreshing itself, since the comparison group isn't "the same pages before and after,"
it's "pages someone bet on" vs "pages nobody bothered with." The paper's own `361+` freshness bucket
(283:1 ratio on n=1 declining page) shows it's already alert to small/biased samples elsewhere -- I'd ask
the same selection question here.

**ML Appendix, "What Predicts Health?"** (p.27). Random Forest feature importance for predicting
health score finds Average Position (43%) and Impressions (32%) as the top two predictors, with the
paper noting this itself: "the target itself is partly constructed from some of these inputs."

**My methodology question:** the paper's own Health Score formula is impressions (30 pts) + position
(30 pts) + ctr (20 pts) + scroll depth (20 pts) (p.5). If two of the three top "predictors" are literally
terms in the score's formula, a holdout split doesn't fix that -- it's not measuring whether the model
learned something about content quality, it's measuring whether the model can recover a known formula
from its own known inputs, which it should do almost perfectly regardless of split. This is the same
leakage shape I have to watch for in my own model: a feature that's label-derived stays label-derived on
a fresh test set. The paper does flag this in one sentence ("importance is descriptive rather than
causal"), which is the right instinct -- I'd just push it further and say this section isn't really
measuring predictive skill at all.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape)

(30000, 44)


In [2]:
NUMERIC_FEATURES = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct", "search_volume", "competition", "cpc",
    "word_count", "char_count",
]
CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

def build_features(frame, columns=None, extra_numeric=None):
    numeric_cols = NUMERIC_FEATURES + (extra_numeric or [])
    X = frame[numeric_cols].copy()
    for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
        X[f"has_{col}"] = X[col].notna().astype(int)
    X = X.fillna(0)
    cats = pd.get_dummies(frame[CATEGORICAL_FEATURES].fillna("unknown"), prefix=CATEGORICAL_FEATURES)
    X = pd.concat([X, cats], axis=1)
    if columns is not None:
        X = X.reindex(columns=columns, fill_value=0)
    return X

def precision_at_k(labels, scores, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

tier_order = ["top_3", "page_1", "striking", "page_3_5", "deep"]

In [3]:
def run_pipeline(train_df, test_df, extra_numeric=None):
    train_df = train_df.copy()
    test_df = test_df.copy()
    for frame in (train_df, test_df):
        frame["is_declining_label"] = (frame["trend_direction"] == "down").astype(int)

    X_train_raw = build_features(train_df, extra_numeric=extra_numeric)
    X_test_raw = build_features(test_df, columns=X_train_raw.columns, extra_numeric=extra_numeric)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_test = scaler.transform(X_test_raw)

    scores = {k: silhouette_score(X_train, KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_train).labels_)
              for k in range(2, 9)}
    best_k = max(scores, key=scores.get)

    km = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X_train)
    train_df["cluster"] = km.labels_
    test_df["cluster"] = km.predict(X_test)
    cluster_rates = train_df.groupby("cluster")["is_declining_label"].mean()
    test_df["model_score"] = test_df["cluster"].map(cluster_rates)

    expected_ctr = (
        train_df[(train_df["impressions_90d"] >= 500) & (train_df["position_tier"].isin(tier_order))]
        .groupby("position_tier")["ctr"].median()
    )
    test_df["expected_ctr"] = test_df["position_tier"].map(expected_ctr)
    test_df["ctr_gap"] = (test_df["expected_ctr"] - test_df["ctr"]).clip(lower=0)
    stale = (test_df["days_since_last_update"] >= 90).astype(int)
    is_visible = (test_df["impressions_90d"] >= 500).astype(int)
    has_position = test_df["position_tier"].isin(tier_order).astype(int)
    test_df["baseline_score"] = stale * is_visible * has_position * test_df["ctr_gap"] * test_df["impressions_90d"]

    rows = []
    for k in (25, 50, 100):
        rows.append({
            "k": k,
            "baseline_precision": precision_at_k(test_df["is_declining_label"].values, test_df["baseline_score"].values, k),
            "model_precision": precision_at_k(test_df["is_declining_label"].values, test_df["model_score"].values, k),
            "base_rate": test_df["is_declining_label"].mean(),
        })
    return best_k, pd.DataFrame(rows), train_df, test_df

In [4]:
# BEFORE: plain random 80/20, same as w05_model.ipynb
train_random, test_random = train_test_split(df, test_size=0.2, random_state=42)
k_random, compare_random, _, _ = run_pipeline(train_random, test_random)
print("random split, best k:", k_random)
compare_random

random split, best k: 2


,k,baseline_precision,model_precision,base_rate
0,25,0.48,0.52,0.544667
1,50,0.56,0.64,0.544667
2,100,0.59,0.60,0.544667


In [5]:
# AFTER: grouped by client_id -- no client appears on both sides
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_grouped, test_grouped = df.iloc[train_idx], df.iloc[test_idx]

shared = set(train_grouped["client_id"]) & set(test_grouped["client_id"])
print("clients in both train and test:", len(shared), "of", df["client_id"].nunique())

k_grouped, compare_grouped, _, _ = run_pipeline(train_grouped, test_grouped)
print("grouped split, best k:", k_grouped)
compare_grouped

clients in both train and test: 0 of 32


grouped split, best k: 2


,k,baseline_precision,model_precision,base_rate
0,25,0.4,0.72,0.510952
1,50,0.5,0.70,0.510952
2,100,0.5,0.64,0.510952


In [6]:
before_after = compare_random.merge(compare_grouped, on="k", suffixes=("_random", "_grouped"))
before_after

,k,baseline_precision_random,model_precision_random,base_rate_random,baseline_precision_grouped,model_precision_grouped,base_rate_grouped
0,25,0.48,0.52,0.544667,0.4,0.72,0.510952
1,50,0.56,0.64,0.544667,0.5,0.70,0.510952
2,100,0.59,0.60,0.544667,0.5,0.64,0.510952


**Before (random 80/20, same as w05):** baseline precision 0.48 / 0.56 / 0.59 at k=25/50/100, model
0.52 / 0.64 / 0.60, base rate 0.545.

**After (grouped by client_id, 0 shared clients):** baseline 0.40 / 0.50 / 0.50, model 0.72 / 0.70 / 0.64,
base rate 0.511.

The model looks better on every k under the grouped split. I don't trust that difference yet. This is a
single GroupShuffleSplit holding out roughly 6 of 32 clients -- a small enough group that which specific
clients land in the test set can swing the numbers on its own, and the base rate moved too (0.545 ->
0.511), so the two "after" and "before" columns aren't even measuring the exact same population. The
honest reading is: the random-split number was probably too optimistic (client leakage was real -- every
client showed up on both sides), but I can't yet tell how much of the grouped-split number is genuine
generalization versus which 6 clients happened to get held out. That would need repeated grouped splits
(GroupKFold), not one, before I'd call it settled.

One more thing worth flagging from w05: at k=25 on the random split, model precision (0.52) sits *below*
the base rate (0.545) -- worse than guessing. That matches what I already found in w05's error analysis:
cluster 0 ties 22,079 pages to the same score, so a small top-k is close to a random draw from that
cluster, not a real ranking.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [7]:
# attack checklist
banned = {"trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
          "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"}
print("[ ] no label-derived columns in features:", banned & set(NUMERIC_FEATURES + CATEGORICAL_FEATURES) == set())
print("[ ] no product flags in this dataset (none shipped):", True)
print("[ ] base rate printed next to every metric: see base_rate column above")
print("[ ] split grouped by repeating entity: done above (client_id)")

[ ] no label-derived columns in features: True
[ ] no product flags in this dataset (none shipped): True
[ ] base rate printed next to every metric: see base_rate column above
[ ] split grouped by repeating entity: done above (client_id)


In [8]:
# deliberately ADD the label-source column as a feature and watch the score jump
k_leaky, compare_leaky, _, _ = run_pipeline(train_grouped, test_grouped, extra_numeric=["trend_pct"])
print("with trend_pct injected, best k:", k_leaky)
compare_leaky

with trend_pct injected, best k: 2


,k,baseline_precision,model_precision,base_rate
0,25,0.4,0.72,0.510952
1,50,0.5,0.70,0.510952
2,100,0.5,0.64,0.510952


In [9]:
# the KMeans injection above didn't move the score -- before reading that as "no leakage risk,"
# check whether the test harness itself can even detect a strong leaky signal.
# score directly by the label-source column, isolated from the other 61 features.
isolated_labels = (test_grouped["trend_direction"] == "down").astype(int).values
isolated_score = -test_grouped["trend_pct"].values  # more negative trend_pct -> more likely declining
for k in (25, 50, 100):
    p = precision_at_k(isolated_labels, isolated_score, k)
    print(f"k={k}: precision using trend_pct alone = {p:.2f}")

k=25: precision using trend_pct alone = 1.00
k=50: precision using trend_pct alone = 1.00
k=100: precision using trend_pct alone = 1.00


Injecting `trend_pct` into the full 62-feature clustering pipeline changed nothing -- identical precision
numbers with or without it. Before reading that as "no leakage risk here," I checked whether my test
harness could even detect a real leaky signal: scoring pages directly by `trend_pct` alone (no clustering,
no other features) gets precision@25/50/100 = 1.00, 1.00, 1.00 -- it's a near-perfect predictor on its
own, since `trend_direction` is literally computed from it.

So the harness isn't broken. What happened is specific to clustering: one column, even a near-perfect one,
gets averaged into a Euclidean distance across 62 scaled dimensions, and K-Means doesn't fit weights the
way a supervised model would -- it has no way to know that particular column deserves more say than the
other 61. A logistic regression given the same injected feature would very likely jump toward precision
1.0 immediately, the way the skill describes. This doesn't mean clustering is safe from leakage in general
-- it means this specific injection test is a weaker check for an unsupervised method than for a
supervised one, and I'd want a sharper test (like the isolated-feature check above) whenever the standard
injection test comes back quiet. My actual feature set never includes `trend_pct` or `trend_direction`, so
this was a probe, not something I need to remove.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Boldest sentence, from `w05_model.ipynb`:** "It wins clearly at k=25 and k=50 (0.52 vs 0.48, 0.64 vs
0.56) because cluster membership alone is doing real work at that scale, but that edge won't hold as k
gets close to cluster size."

**Rewrite, in safe language:** Observed: on one random 80/20 split, the cluster-based ranking's
precision@25 and precision@50 (0.52, 0.64) measured higher than the rule baseline's (0.48, 0.56).
Directional, not conclusive: a grouped-by-client split on the same data changed both the base rate and
the gap between the two methods, so this comparison is sensitive to which clients land in the held-out
set. Decision-support read: this is an early signal that the cluster-based ranking may out-perform the
baseline, not a settled result -- it would need repeated grouped splits before either number should guide
an actual review queue.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.